In [ ]:
// hide-on-readme
// prelude

import {updateNotebookToc, addPreCommitHook} from './scripts/prelude'

addPreCommitHook();
updateNotebookToc();

# LMPriestley's Notebook

This is a personal project to collect my personal knowledge and small utilities into an obsidian inspired jupyter notebook.

To view the interactive version of this project, please visit [lmpriestley.com](https://freyground.com). Sources available at [github.com/freylint/gbnotebook](https://www.github.com/freylint/gbnotebook).

## Table of Contents

- [License](#License)
- [Packages](#Packages)
  - [NBECS Interface - Notebook ECS interfaces](#NBECS-Interface---Notebook-ECS-interfaces)
  - [NBECS](#NBECS)
- [Articles](#Articles)
- [Notes](#Notes)
- [Images](#Images)
- [Homelab Setup](#Homelab-Setup)
  - [Hardware](#Hardware)
    - [Network](#Network)
    - [Homelab Server](#Homelab-Server)
    - [Thin Client](#Thin-Client)


## License

This project is licensed under the GNU Affero General Public License v3.0.
See [LICENSE](LICENSE) for the full text.

## Homelab Setup

### Hardware
#### Network
Commodity home router into a switch.
#### Homelab Server
4u threadripper machine w/ 2x dedicated GPU and 128GB of RAM.
#### Thin Client
Commodity x86 laptops w/ port replicators.


In [2]:
// Homelab BaseImage Generation script
import {
  DEBIAN_RELEASE,
  DEBOOTSTRAP_VARIANT,
  DEBOOTSTRAP_MIRROR,
  LOCAL_DIST_DIR_SRV,
  LOCAL_DIST_DIR_CLIENT,
  LOCAL_BUILD_DIR,
  runSyncCommand,
  runChrootCommand,
  ensureDir,
  isDirEmpty,
  clearDir,
  checkCommandAvailable,
} from './scripts/prelude';

import {existsSync} from 'fs';

function buildBaseImage() {
    checkCommandAvailable('debootstrap');
    ensureDir(LOCAL_DIST_DIR_SRV);

    if (existsSync(LOCAL_BUILD_DIR) && !isDirEmpty(LOCAL_BUILD_DIR)) {
        console.log('Skipping debootstrap because target directory already exists and is non-empty:', LOCAL_BUILD_DIR);
        return;
    }

    clearDir(LOCAL_BUILD_DIR);

    console.log('Syncing Debian min image for test build...');
    console.log('Debian release:', DEBIAN_RELEASE);
    console.log('Debootstrap variant:', DEBOOTSTRAP_VARIANT);
    console.log('Output directory:', LOCAL_BUILD_DIR);
    console.log('Mirror:', DEBOOTSTRAP_MIRROR);

    runSyncCommand('debootstrap', [
        '--variant=' + DEBOOTSTRAP_VARIANT,
        DEBIAN_RELEASE,
        LOCAL_BUILD_DIR,
        DEBOOTSTRAP_MIRROR,
    ]);

    console.log('Debootstrap command completed.');
}
function buildClientImage() {
    // RSYNC base to client directory
    ensureDir(LOCAL_DIST_DIR_CLIENT);
    runSyncCommand('rsync', ['-a', LOCAL_BUILD_DIR, LOCAL_DIST_DIR_CLIENT]);

    runChrootCommand(LOCAL_DIST_DIR_CLIENT, 'echo', ['Hello from chroot']); 
}

function buildServerImage() {
    // RSYNC base to server directory
    ensureDir(LOCAL_DIST_DIR_SRV);
    runSyncCommand('rsync', ['-a', LOCAL_BUILD_DIR, LOCAL_DIST_DIR_SRV]);

    runChrootCommand(LOCAL_DIST_DIR_SRV, 'echo', ['Hello from server chroot']);
}

function buildHLImages() {
    buildBaseImage();
    buildClientImage();
    buildServerImage();


    // TODO Setup OSTREE Repository for managing image versions
}

buildHLImages();



Syncing Debian min image for test build...
Debian release: testing
Debootstrap variant: minbase
Output directory: /workspaces/graphnb/build/base/
Mirror: http://deb.debian.org/debian
I: Retrieving InRelease 
I: Checking Release signature
I: Valid Release signature (key id B8E5F13176D2A7A75220028078DBA3BC47EF2265)
I: Retrieving Packages 
I: Validating Packages 
I: Resolving dependencies of required packages...
I: Resolving dependencies of base packages...
I: Checking component main on http://deb.debian.org/debian...
I: Retrieving apt 3.2.0
I: Validating apt 3.2.0
I: Retrieving base-files 14
I: Validating base-files 14
I: Retrieving base-passwd 3.6.8
I: Validating base-passwd 3.6.8
I: Retrieving bash 5.3-2
I: Validating bash 5.3-2
I: Retrieving bsdutils 1:2.42-4
I: Validating bsdutils 1:2.42-4
I: Retrieving coreutils 9.10-1
I: Validating coreutils 9.10-1
I: Retrieving dash 0.5.12-12
I: Validating dash 0.5.12-12
I: Retrieving debconf 1.5.92
I: Validating debconf 1.5.92
I: Retrieving debia

## Articles